# AuscultaForge — CirCor Patient-Level Murmur MIL · Training Lab v1

**Kalıcı araştırma altyapısı / gerçek veri üzerinde eğitim.** Tek hastaya ait farklı auskültasyon kayıtları attention-MIL ile birlikte değerlendirilir. Bu notebook, [PhysioNet CirCor v1.0.3](https://physionet.org/content/circor-heart-sound/1.0.3/) kamuya açık eğitim alt kümesini kullanır.

> **Research only** — Üfürüm benzeri akustik örüntüler için araştırma sınıflandırmasıdır. Kalp hastalığı, aritmi veya klinik teşhis yapmaz. ESP32-S3 fiziksel doğrulamasıyla karıştırılmamalıdır.

## Çalıştırma
1. **Runtime → Change runtime type → GPU (if available)**. GPU ve 12 saat garanti değildir.
2. Notebook açıkken en üstten sırayla hücreleri çalıştır. İlk hücre **AuscultaForge_ML_RepoDrop.zip** dosyasını isteyecek.
3. Google Drive mount izni ver. Checkpoint'ler doğrudan Drive'a yazılır; bağlantı kesilirse son epoch'tan devam edebilirsin.
4. PhysioNet'in 559 MB civarındaki veri paketini indirip ön işlemeden sonra **gerçek eğitimi** başlat.
5. Veriyi veya bireysel ses kayıtlarını GitHub'a commit etme. Modelin başarısını ancak gerçek `metrics.json` üretildikten sonra konuşuruz.


In [ ]:
# CELL 1 — Upload the permanent repo-ready ML package
from google.colab import files
from pathlib import Path
import zipfile, io, os, subprocess, sys
uploaded = files.upload()   # select AuscultaForge_ML_RepoDrop.zip
archives = [name for name in uploaded if name.endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Please upload exactly one AuscultaForge_ML_RepoDrop.zip archive')
BASE = Path('/content/ausculta_lab')
BASE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    # Secure: ensure archives cannot write outside /content/ausculta_lab.
    for member in archive.infolist():
        resolved = (BASE / member.filename).resolve()
        if not resolved.is_relative_to(BASE.resolve()):
            raise ValueError('Unsafe archive path: ' + member.filename)
    archive.extractall(BASE)
PROJECT = BASE / 'research' / 'ml'
assert (PROJECT / 'ausculta_ml' / 'train.py').exists(), 'Incorrect ZIP layout'
os.chdir(PROJECT)
print('AuscultaForge ML code loaded at:', PROJECT)


In [ ]:
# CELL 2 — Runtime + dependencies
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt'),'awscli','pytest'], check=True)
import torch
print('Python:', sys.version.split()[0], '| Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA; CPU will be slower')
!python -m pytest tests -q


In [ ]:
# CELL 3 — Google Drive: durable training checkpoints
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/AuscultaForge_ML')
RUN = DRIVE_ROOT/'runs'/'circor_mil_v1'
ARTIFACT = DRIVE_ROOT/'artifacts'/'circor_mil_v1'
RUN.mkdir(parents=True, exist_ok=True)
print('Persistent run path:', RUN)


### Veri kaynağı — açık erişim ve atıf

CirCor v1.0.3, **ODC-By v1.0** lisansıyla PhysioNet üzerinde yayımlanmıştır. Metaveri ve WAV'ları resmi PhysioNet açık S3 aynasından indiriyoruz; **özel hasta verisi yok**, veri eğitim kodunun içinde gelmiyor. İndirme ve Colab erişimi için internet gerekir.

Kaynak: Oliveira et al., CirCor DigiScope Dataset v1.0.3, DOI: `10.13026/tshs-mw03`. Murmur etiketi **hasta düzeyindedir**. Unknown etiketi pozitif/negatife zorlanmadan dışlanır.


In [ ]:
# CELL 4 — Download authentic CirCor 1.0.3 dataset (no mock/synthetic shortcuts)
import subprocess
RAW = BASE/'data'/'raw'/'circor'/'1.0.3'
RAW.mkdir(parents=True, exist_ok=True)
if not ((RAW/'training_data.csv').exists() and (RAW/'training_data').is_dir()):
    cmd = ['aws','s3','sync','--no-sign-request',
           's3://physionet-open/circor-heart-sound/1.0.3/',str(RAW)]
    print('Downloading official public dataset:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
assert (RAW/'training_data.csv').exists(), 'Dataset CSV missing after download'
assert any((RAW/'training_data').glob('*.wav')), 'WAV files not found after download'
print('Found WAVs:', len(list((RAW/'training_data').glob('*.wav'))))


In [ ]:
# CELL 5 — Prepare immutable linked-patient splits and spectrogram features
import json, shutil
from ausculta_ml.data import make_index, frozen_split, prepare_spectrogram_cache
import yaml
CFG = PROJECT/'configs'/'circor_mil_v1.yaml'
config = yaml.safe_load(CFG.read_text())
WORK = BASE/'work'/'circor'
WORK.mkdir(parents=True, exist_ok=True)
if (RUN/'split_manifest.json').exists():
    shutil.copy2(RUN/'split_manifest.json', WORK/'split_manifest.json')
records = make_index(RAW, WORK/'circor_index.json')
splits = frozen_split(records, WORK/'split_manifest.json',seed=config['seed'],
                      val_fraction=config['val_fraction'],test_fraction=config['test_fraction'])
shutil.copy2(WORK/'split_manifest.json', RUN/'split_manifest.json')
for name, rows in splits.items():
    print(name, 'patient IDs:', len(rows), '| murmur present:', sum(r['label'] for r in rows))
prepare_spectrogram_cache(records,WORK/'spectrogram_cache',sample_rate=config['sample_rate'],
         nfft=config['stft_n_fft'],hop=config['stft_hop'],
         low_hz=config['band_low_hz'],high_hz=config['band_high_hz'])


In [ ]:
# CELL 6 — Actual full-data training. 35 epochs maximum, early stopping & resume.
# Changes to config after training starts invalidate the resume signature: intentional.
import subprocess, sys, torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU not available. Set Runtime → Change runtime type → T4 GPU before full training.')
command=[sys.executable,'-m','ausculta_ml.train',
         '--config',str(CFG),'--raw',str(RAW),'--work',str(WORK),'--run',str(RUN)]
if (RUN/'checkpoint_last.pt').exists():
    command.append('--resume')
print('Running full subject-level training; checkpoints saved every epoch to Drive.')
print('Command:', ' '.join(command))
subprocess.run(command,cwd=PROJECT,check=True)


In [ ]:
# CELL 7 — Inspect REAL results (do not assume an accuracy before training)
import json, pandas as pd
report=json.loads((RUN/'metrics.json').read_text())
print('Validation:',report['validation'])
print('Internal heldout test:',report['internal_heldout_test'])
print('Best epoch:',report['best_epoch'])
print('Clinical validation: NOT EXECUTED')
log=pd.read_json(RUN/'history.jsonl',lines=True)
display(log.tail(10))
ax=log.plot(x='epoch',y=['val_auc'],title='AuscultaForge ML — patient heldout validation AUROC')


In [ ]:
# CELL 8 — Export a versioned CPU TorchScript research model
import subprocess,sys
subprocess.run([sys.executable,'-m','ausculta_ml.export','--run',str(RUN),'--out',str(ARTIFACT)],
               cwd=PROJECT,check=True)
print((ARTIFACT/'manifest.json').read_text())


In [ ]:
# CELL 9 — Non-destructive AI Filter Advisor engineering foundation
# This is deterministic signal analysis, NOT a learned noise classifier.
import numpy as np
from ausculta_ml.filter_advisor import recommend_filters, render_filter_preview
# Replace with a properly authorized research WAV path for your own study.
example_fs=2000
time=np.arange(example_fs*3)/example_fs
example=0.1*np.sin(2*np.pi*95*time) + 0.03*np.sin(2*np.pi*50*time)
suggestions=recommend_filters(example,example_fs)
for suggestion in suggestions:
    print(suggestion)
print('Original waveform remains unchanged; recommended filters require acceptance and effect evaluation.')


## Sonraki geliştirmeler (aynı altyapı, sıfırdan başlamak yok)

- `quality_v1`: gerçek artefakt etiketleri / açıkça tanımlı kontrollü bozulmalar üzerinde model
- `filter_optimizer_v1`: birden fazla güvenli filtre adayını *referans bozulması + morfoloji koruma* ölçütleriyle karşılaştıran değerlendirme
- `segmentation_v1`: S1/sistol/S2/diyastol tahmini; semantik olay metrikleri
- `denoiser_v1`: paired clean/corrupt veri, güvenilir sinyal bozunum metrikleri; halüsinasyon uyarısı
- `domain_shift_v1`: klinik veri ↔ phantom/INMP441/ECM değişim analizi

**Bu notebook yalnızca ilk eğitilen başlığı gerçekleştirir.** Fiziksel G1, klinik tanı, otomatik filtre aktivasyonu veya bağımsız genel geçer doğruluk iddiası yoktur.
